# Two-Stage GNN Residual Training

**Stage 1:** jointly train LoRA and the GNN residual branch through synthetic Levels 1–5.  
**Stage 2:** load the final Level 5 LoRA + GNN checkpoint and continue training both components on WikiTableQuestions.

The Qwen backbone remains frozen in both stages. Every checkpoint is written directly to Google Drive and both stages resume safely after a runtime disconnect.

## 1. GPU and repository setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda} | available={torch.cuda.is_available()}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Git commit: {commit}")
%cd /content/table-cnn-mrc

## 2. Install dependencies and mount Drive

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

## 3. Verify structured synthetic tables

In [ ]:
from src.synthetic_curriculum import load_all_synthetic_mrc_levels

synthetic_levels = load_all_synthetic_mrc_levels(REPO_DIR)
for level, examples in synthetic_levels.items():
    example = examples[0]
    print(
        f"Level {level}: {len(examples)} examples | "
        f"table={len(example['table']['rows']) + 1}x{len(example['table']['header'])}"
    )
print("\nExample question:", synthetic_levels[1][0]["question"])
print("Example answer:", synthetic_levels[1][0]["answers"])
print("Example table:", synthetic_levels[1][0]["table"])
del synthetic_levels

## 4. Training configuration

In [ ]:
SYNTHETIC_LEVELS = [1, 2, 3, 4, 5]
SYNTHETIC_EPOCHS = [3, 3, 3, 3, 3]
SYNTHETIC_LEARNING_RATE = 1e-4
WTQ_EPOCHS = 10
WTQ_LEARNING_RATE = 1e-4
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
CHECKPOINT_EVERY_STEPS = 25
WTQ_EARLY_STOPPING_PATIENCE = 1
SEED = 42

DRIVE_ROOT = Path("/content/drive/MyDrive/cnn_qwen_table_mcr")
OUTPUT_ROOT = DRIVE_ROOT / "outputs/gnn_synthetic_pretrain_wtq"
OFFICIAL_CACHE = DRIVE_ROOT / "outputs/diagnostics/wtq_official_1.0.2"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if len(SYNTHETIC_LEVELS) != len(SYNTHETIC_EPOCHS):
    raise ValueError("SYNTHETIC_LEVELS and SYNTHETIC_EPOCHS must match")
print(f"Synthetic schedule: {dict(zip(SYNTHETIC_LEVELS, SYNTHETIC_EPOCHS))}")
print(f"Direct Drive output: {OUTPUT_ROOT}")

## 5. Gradient smoke test

This verifies one real WTQ forward/backward pass, token-to-cell alignment, a GNN gate gradient, and a LoRA gradient before the full run.

In [ ]:
smoke_command = [
    sys.executable, "-u", str(REPO_DIR / "scripts/smoke_test.py"),
    "--config", str(REPO_DIR / "configs/gnn_residual_relational_early.yaml"),
]
subprocess.run(smoke_command, cwd=REPO_DIR, check=True)

## 6. Run both stages

Stage 1 evaluates WTQ validation at Base and after every synthetic level. Stage 2 then uses the final Level 5 checkpoint—not the validation-selected synthetic checkpoint—and fine-tunes the same LoRA + GNN model on the full WTQ training split.

In [ ]:
command = [
    sys.executable, "-u", str(REPO_DIR / "scripts/run_gnn_two_stage.py"),
    "--data-root", str(REPO_DIR),
    "--output-root", str(OUTPUT_ROOT),
    "--official-cache-dir", str(OFFICIAL_CACHE),
    "--levels", *[str(value) for value in SYNTHETIC_LEVELS],
    "--synthetic-epochs", *[str(value) for value in SYNTHETIC_EPOCHS],
    "--synthetic-learning-rate", str(SYNTHETIC_LEARNING_RATE),
    "--wtq-epochs", str(WTQ_EPOCHS),
    "--wtq-learning-rate", str(WTQ_LEARNING_RATE),
    "--batch-size", str(BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--checkpoint-every-steps", str(CHECKPOINT_EVERY_STEPS),
    "--wtq-early-stopping-patience", str(WTQ_EARLY_STOPPING_PATIENCE),
    "--seed", str(SEED),
]
command_text = " ".join(shlex.quote(str(part)) for part in command)
status_path = Path("/tmp/gnn_two_stage_exit_code.txt")
status_path.unlink(missing_ok=True)
shell_command = (
    f"cd {shlex.quote(str(REPO_DIR))} && "
    f"PYTHONUNBUFFERED=1 TABLE_MRC_PLAIN_PROGRESS=1 "
    f"TABLE_MRC_PLAIN_LOG_EVERY=100 TQDM_DISABLE=1 "
    f"{command_text}; printf '%s' $? > {shlex.quote(str(status_path))}"
)
get_ipython().system(shell_command)
if not status_path.is_file():
    raise RuntimeError("Two-stage exit status was not recorded")
return_code = int(status_path.read_text(encoding="utf-8").strip())
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)

## 7. Results

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

stage_1_results = pd.read_csv(
    OUTPUT_ROOT / "stage_1_synthetic/results/curriculum_results.csv"
)
display(stage_1_results)
stage_1_results.plot(
    x="stage", y="wtq_validation_score", marker="o", figsize=(8, 4),
    title="GNN synthetic-pretraining transfer curve", legend=False,
)
plt.ylabel("WTQ validation denotation accuracy")
plt.grid(alpha=0.3)
plt.show()

summary_path = OUTPUT_ROOT / "two_stage_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))

In [ ]:
wtq_history_path = OUTPUT_ROOT / "stage_2_wtq/history.json"
wtq_history = json.loads(wtq_history_path.read_text(encoding="utf-8"))
wtq_only_history_path = DRIVE_ROOT / "outputs/gnn_residual_relational_early/history.json"
wtq_only_gnn_score = None
if wtq_only_history_path.is_file():
    wtq_only_history = json.loads(wtq_only_history_path.read_text(encoding="utf-8"))
    wtq_only_gnn_score = wtq_only_history.get("best_metric")
    print(f"Existing WTQ-only GNN baseline: {wtq_only_gnn_score}")
wtq_rows = []
for record in wtq_history.get("epochs", []):
    validation = record.get("validation", {})
    if "denotation_accuracy" in validation:
        wtq_rows.append({
            "epoch": record["epoch"],
            "training_loss": record["training_loss"],
            "validation_denotation_accuracy": validation["denotation_accuracy"],
        })
wtq_results = pd.DataFrame(wtq_rows)
display(wtq_results)
if not wtq_results.empty:
    ax = wtq_results.plot(
        x="epoch", y="validation_denotation_accuracy", marker="o",
        figsize=(8, 4), title="WTQ fine-tuning after GNN synthetic pretraining",
    )
    ax.axhline(0.514659, color="gray", linestyle="--", label="serialized LoRA baseline: 0.5147")
    if wtq_only_gnn_score is not None:
        ax.axhline(wtq_only_gnn_score, color="tab:red", linestyle=":", label=f"WTQ-only GNN: {wtq_only_gnn_score:.4f}")
    ax.set_ylabel("WTQ validation denotation accuracy")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()